<a href="https://colab.research.google.com/github/Tiagolopezbit/Tiago-Lopez-IA/blob/main/RAG_Bot_Ventas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Bot RAG con Pandas
### Genera respuestas inteligentes a partir de tus datos de ventas

**Flujo del sistema:**
```
ventas.csv → Pandas (normalización) → Corpus → Embeddings → RAG → Respuesta
```

**Requisitos:**
- Google Colab (con GPU opcional)
- API Key de OpenAI o modelo local (usaremos sentence-transformers gratuito)


## 📦 Paso 1: Instalación de dependencias

In [3]:
!pip install -q sentence-transformers faiss-cpu pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 63.3 MB/s eta 0:00:00


## 📊 Paso 2: Crear datos de ventas de ejemplo (ventas.csv)

In [4]:
import pandas as pd
import numpy as np
from io import StringIO

# Datos de ejemplo similares a ventas.csv
data_csv = """fecha,producto,categoria,cantidad,precio_unitario,vendedor,region
2024-01-15,Laptop Pro,Tecnología,3,1500.00,Ana López,Norte
2024-01-16,Mouse Inalámbrico,Accesorios,25,29.99,Carlos Ruiz,Sur
2024-01-17,Monitor 4K,Tecnología,5,450.00,Ana López,Norte
2024-01-18,Teclado Mecánico,Accesorios,10,89.50,María García,Este
2024-01-19,Laptop Pro,Tecnología,2,1500.00,Luis Martínez,Oeste
2024-01-20,Auriculares BT,Accesorios,15,120.00,Carlos Ruiz,Sur
2024-02-01,Monitor 4K,Tecnología,8,450.00,María García,Este
2024-02-03,Webcam HD,Accesorios,20,75.00,Ana López,Norte
2024-02-10,Laptop Pro,Tecnología,1,1500.00,Luis Martínez,Oeste
2024-02-15,Mouse Inalámbrico,Accesorios,30,29.99,Carlos Ruiz,Sur
2024-03-01,Tablet 10',Tecnología,6,350.00,María García,Este
2024-03-05,Laptop Pro,Tecnología,4,1550.00,Ana López,Norte
2024-03-10,Auriculares BT,Accesorios,12,120.00,Luis Martínez,Oeste
2024-03-20,Monitor 4K,Tecnología,3,460.00,Carlos Ruiz,Sur
2024-03-25,Webcam HD,Accesorios,18,75.00,María García,Este"""

# Cargar datos
df = pd.read_csv(StringIO(data_csv))
print("✅ Datos cargados exitosamente")
print(f"📋 Filas: {len(df)} | Columnas: {len(df.columns)}")
df.head(10)

✅ Datos cargados exitosamente
📋 Filas: 15 | Columnas: 7


,fecha,producto,categoria,cantidad,precio_unitario,vendedor,region
0,2024-01-15,Laptop Pro,Tecnología,3,1500.00,Ana López,Norte
1,2024-01-16,Mouse Inalámbrico,Accesorios,25,29.99,Carlos Ruiz,Sur
2,2024-01-17,Monitor 4K,Tecnología,5,450.00,Ana López,Norte
3,2024-01-18,Teclado Mecánico,Accesorios,10,89.50,María García,Este
4,2024-01-19,Laptop Pro,Tecnología,2,1500.00,Luis Martínez,Oeste
5,2024-01-20,Auriculares BT,Accesorios,15,120.00,Carlos Ruiz,Sur
6,2024-02-01,Monitor 4K,Tecnología,8,450.00,María García,Este
7,2024-02-03,Webcam HD,Accesorios,20,75.00,Ana López,Norte
8,2024-02-10,Laptop Pro,Tecnología,1,1500.00,Luis Martínez,Oeste
9,2024-02-15,Mouse Inalámbrico,Accesorios,30,29.99,Carlos Ruiz,Sur


## 🧹 Paso 3: Normalización y limpieza de datos con Pandas

In [5]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

class NormalizadorVentas:
    """Clase que normaliza y limpia el DataFrame de ventas"""

    def __init__(self, df):
        self.df_original = df.copy()
        self.df = df.copy()
        self.scaler = MinMaxScaler()
        self.encoders = {}

    def limpiar(self):
        """Limpia valores nulos y formatos"""
        # Convertir fecha a datetime
        self.df['fecha'] = pd.to_datetime(self.df['fecha'])

        # Eliminar duplicados
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        print(f"🗑️  Duplicados eliminados: {before - len(self.df)}")

        # Rellenar nulos con mediana/moda
        for col in self.df.select_dtypes(include='number').columns:
            self.df[col] = self.df[col].fillna(self.df[col].median())
        for col in self.df.select_dtypes(include='object').columns:
            self.df[col] = self.df[col].fillna(self.df[col].mode()[0])

        print("✅ Limpieza completada")
        return self

    def enriquecer(self):
        """Crea columnas derivadas útiles"""
        self.df['total_venta'] = self.df['cantidad'] * self.df['precio_unitario']
        self.df['mes'] = self.df['fecha'].dt.month_name().str.capitalize()
        self.df['trimestre'] = self.df['fecha'].dt.quarter.map({1:'Q1',2:'Q2',3:'Q3',4:'Q4'})
        print("✅ Columnas enriquecidas: total_venta, mes, trimestre")
        return self

    def normalizar_numericas(self):
        """Normaliza columnas numéricas (0 a 1) con MinMaxScaler"""
        cols = ['cantidad', 'precio_unitario', 'total_venta']
        self.df[[c + '_norm' for c in cols]] = self.scaler.fit_transform(self.df[cols])
        print(f"✅ Normalizadas con MinMaxScaler: {cols}")
        return self

    def codificar_categoricas(self):
        """Codifica variables categóricas con LabelEncoder"""
        cols_cat = ['categoria', 'region']
        for col in cols_cat:
            le = LabelEncoder()
            self.df[col + '_cod'] = le.fit_transform(self.df[col])
            self.encoders[col] = le
        print(f"✅ Codificadas con LabelEncoder: {cols_cat}")
        return self

    def obtener_df(self):
        return self.df


# Ejecutar normalización
norm = NormalizadorVentas(df)
norm.limpiar().enriquecer().normalizar_numericas().codificar_categoricas()
df_norm = norm.obtener_df()

print("\n📊 DataFrame normalizado:")
df_norm.head()

🗑️  Duplicados eliminados: 0
✅ Limpieza completada
✅ Columnas enriquecidas: total_venta, mes, trimestre
✅ Normalizadas con MinMaxScaler: ['cantidad', 'precio_unitario', 'total_venta']
✅ Codificadas con LabelEncoder: ['categoria', 'region']

📊 DataFrame normalizado:


,fecha,producto,categoria,cantidad,precio_unitario,vendedor,region,total_venta,mes,trimestre,cantidad_norm,precio_unitario_norm,total_venta_norm,categoria_cod,region_cod
0,2024-01-15,Laptop Pro,Tecnología,3,1500.00,Ana López,Norte,4500.00,January,Q1,0.068966,0.967105,0.688088,1,1
1,2024-01-16,Mouse Inalámbrico,Accesorios,25,29.99,Carlos Ruiz,Sur,749.75,January,Q1,0.827586,0.000000,0.000000,0,3
2,2024-01-17,Monitor 4K,Tecnología,5,450.00,Ana López,Norte,2250.00,January,Q1,0.137931,0.276321,0.275263,1,1
3,2024-01-18,Teclado Mecánico,Accesorios,10,89.50,María García,Este,895.00,January,Q1,0.310345,0.039151,0.026650,0,0
4,2024-01-19,Laptop Pro,Tecnología,2,1500.00,Luis Martínez,Oeste,3000.00,January,Q1,0.034483,0.967105,0.412871,1,2


## 📚 Paso 4: Crear el Corpus de conocimiento

In [6]:
class GeneradorCorpus:
    """
    Convierte el DataFrame a texto legible (corpus)
    para que el modelo RAG pueda entenderlo.
    """

    def __init__(self, df):
        self.df = df
        self.corpus = []

    def generar_documentos_por_fila(self):
        """Un documento de texto por cada venta"""
        for _, row in self.df.iterrows():
            doc = (
                f"Venta del {row['fecha'].strftime('%d de %B del %Y')}: "
                f"Se vendieron {int(row['cantidad'])} unidades de '{row['producto']}' "
                f"(categoría {row['categoria']}) a ${row['precio_unitario']:.2f} cada una. "
                f"Total de la venta: ${row['total_venta']:.2f}. "
                f"Vendedor: {row['vendedor']}, región: {row['region']}. "
                f"Trimestre: {row['trimestre']}."
            )
            self.corpus.append(doc)
        return self

    def generar_resumen_por_producto(self):
        """Resumen agregado por producto"""
        resumen = self.df.groupby('producto').agg(
            total_vendido=('cantidad','sum'),
            ingresos_totales=('total_venta','sum'),
            precio_promedio=('precio_unitario','mean'),
            num_transacciones=('fecha','count')
        ).reset_index()

        for _, row in resumen.iterrows():
            doc = (
                f"Resumen del producto '{row['producto']}': "
                f"Total de {int(row['total_vendido'])} unidades vendidas en "
                f"{int(row['num_transacciones'])} transacciones. "
                f"Ingresos totales: ${row['ingresos_totales']:.2f}. "
                f"Precio promedio: ${row['precio_promedio']:.2f}."
            )
            self.corpus.append(doc)
        return self

    def generar_resumen_por_vendedor(self):
        """Resumen de desempeño por vendedor"""
        resumen = self.df.groupby('vendedor').agg(
            ingresos=('total_venta','sum'),
            ventas=('fecha','count')
        ).reset_index().sort_values('ingresos', ascending=False)

        for _, row in resumen.iterrows():
            doc = (
                f"Desempeño del vendedor {row['vendedor']}: "
                f"Realizó {int(row['ventas'])} ventas con ingresos totales de "
                f"${row['ingresos']:.2f}."
            )
            self.corpus.append(doc)
        return self

    def obtener_corpus(self):
        return self.corpus


# Generar corpus
gen = GeneradorCorpus(df_norm)
gen.generar_documentos_por_fila()\
   .generar_resumen_por_producto()\
   .generar_resumen_por_vendedor()
corpus = gen.obtener_corpus()

print(f"✅ Corpus generado: {len(corpus)} documentos")
print("\n📄 Ejemplo de documento:")
print(corpus[0])
print("\n📄 Ejemplo de resumen por producto:")
print(corpus[len(df_norm)])

✅ Corpus generado: 26 documentos

📄 Ejemplo de documento:
Venta del 15 de January del 2024: Se vendieron 3 unidades de 'Laptop Pro' (categoría Tecnología) a $1500.00 cada una. Total de la venta: $4500.00. Vendedor: Ana López, región: Norte. Trimestre: Q1.

📄 Ejemplo de resumen por producto:
Resumen del producto 'Auriculares BT': Total de 27 unidades vendidas en 2 transacciones. Ingresos totales: $3240.00. Precio promedio: $120.00.


## 🔍 Paso 5: Embeddings y búsqueda vectorial con FAISS

In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("⏳ Cargando modelo de embeddings (primera vez puede tardar ~1 min)...")

# Modelo multilingüe gratuito, funciona bien en español
modelo_embed = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("✅ Modelo cargado")
print("⏳ Generando embeddings del corpus...")

# Generar embeddings
embeddings = modelo_embed.encode(corpus, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

# Crear índice FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"\n✅ Índice FAISS creado")
print(f"   Documentos indexados: {index.ntotal}")
print(f"   Dimensión de vectores: {dimension}")

⏳ Cargando modelo de embeddings (primera vez puede tardar ~1 min)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Modelo cargado
⏳ Generando embeddings del corpus...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Índice FAISS creado
   Documentos indexados: 26
   Dimensión de vectores: 384


## 🤖 Paso 6: Motor RAG — recuperación + generación de respuestas

In [8]:
class BotRAG:
    """
    Bot RAG (Retrieval-Augmented Generation) para responder
    preguntas sobre los datos de ventas.

    Arquitectura:
      Pregunta → Embedding → FAISS (top-k docs) → Contexto → Respuesta
    """

    def __init__(self, corpus, index, modelo_embed, df, top_k=3):
        self.corpus = corpus
        self.index = index
        self.modelo_embed = modelo_embed
        self.df = df
        self.top_k = top_k

    def recuperar_contexto(self, pregunta):
        """Recupera los documentos más relevantes del corpus"""
        q_emb = self.modelo_embed.encode([pregunta]).astype('float32')
        distancias, indices = self.index.search(q_emb, self.top_k)
        docs = [self.corpus[i] for i in indices[0]]
        return docs, distancias[0]

    def generar_respuesta(self, pregunta, docs):
        """
        Genera respuesta basada en el contexto recuperado.
        Esta función combina RAG con lógica de análisis de pandas
        para preguntas analíticas concretas.
        """
        p = pregunta.lower()

        # --- Respuestas analíticas directas con pandas ---
        if any(w in p for w in ['mejor', 'top', 'más vendido', 'mas vendido']):
            if 'producto' in p:
                top = self.df.groupby('producto')['total_venta'].sum().idxmax()
                total = self.df.groupby('producto')['total_venta'].sum().max()
                return f"📦 El producto más vendido (en ingresos) es **{top}** con un total de **${total:,.2f}**."

            if 'vendedor' in p:
                top = self.df.groupby('vendedor')['total_venta'].sum().idxmax()
                total = self.df.groupby('vendedor')['total_venta'].sum().max()
                return f"🏆 El mejor vendedor es **{top}** con ingresos de **${total:,.2f}**."

        if any(w in p for w in ['total', 'ingreso', 'revenue', 'facturado']):
            total = self.df['total_venta'].sum()
            return f"💰 El total de ingresos en el período es **${total:,.2f}**."

        if 'region' in p or 'región' in p:
            resumen = self.df.groupby('region')['total_venta'].sum().sort_values(ascending=False)
            resp = "🗺️ **Ingresos por región:**\n"
            for reg, val in resumen.items():
                resp += f"  • {reg}: ${val:,.2f}\n"
            return resp

        if 'categoria' in p or 'categoría' in p:
            resumen = self.df.groupby('categoria')['total_venta'].sum().sort_values(ascending=False)
            resp = "🏷️ **Ingresos por categoría:**\n"
            for cat, val in resumen.items():
                resp += f"  • {cat}: ${val:,.2f}\n"
            return resp

        # --- Respuesta RAG general basada en contexto recuperado ---
        contexto = "\n".join([f"- {d}" for d in docs])
        respuesta = (
            f"🔍 **Basado en los datos más relevantes encontrados:**\n\n"
            f"{contexto}\n\n"
            f"💡 *Si deseas análisis más específicos, prueba preguntar por: "
            f"'mejor producto', 'mejor vendedor', 'total ingresos', 'ingresos por región'.*"
        )
        return respuesta

    def preguntar(self, pregunta):
        """Función principal: recibe pregunta y devuelve respuesta"""
        print(f"\n{'='*60}")
        print(f"❓ Pregunta: {pregunta}")
        print(f"{'='*60}")

        docs, distancias = self.recuperar_contexto(pregunta)

        print(f"\n📚 Documentos recuperados (top {self.top_k}):")
        for i, (doc, dist) in enumerate(zip(docs, distancias), 1):
            print(f"  [{i}] (dist={dist:.2f}) {doc[:80]}...")

        respuesta = self.generar_respuesta(pregunta, docs)
        print(f"\n🤖 Respuesta del Bot:")
        print(respuesta)
        return respuesta


# Instanciar el bot
bot = BotRAG(
    corpus=corpus,
    index=index,
    modelo_embed=modelo_embed,
    df=df_norm,
    top_k=3
)

print("✅ Bot RAG listo para responder preguntas!")

✅ Bot RAG listo para responder preguntas!


## 💬 Paso 7: ¡Hacer preguntas al bot!

In [9]:
# Prueba con varias preguntas
preguntas = [
    "¿Cuál es el mejor producto en ventas?",
    "¿Cuál es el mejor vendedor?",
    "¿Cuál es el total de ingresos?",
    "¿Cómo están los ingresos por región?",
    "¿Cuántas ventas se hicieron de Laptop Pro?",
]

for pregunta in preguntas:
    bot.preguntar(pregunta)


❓ Pregunta: ¿Cuál es el mejor producto en ventas?

📚 Documentos recuperados (top 3):
  [1] (dist=18.90) Venta del 15 de February del 2024: Se vendieron 30 unidades de 'Mouse Inalámbric...
  [2] (dist=19.12) Venta del 18 de January del 2024: Se vendieron 10 unidades de 'Teclado Mecánico'...
  [3] (dist=19.25) Venta del 16 de January del 2024: Se vendieron 25 unidades de 'Mouse Inalámbrico...

🤖 Respuesta del Bot:
📦 El producto más vendido (en ingresos) es **Laptop Pro** con un total de **$15,200.00**.

❓ Pregunta: ¿Cuál es el mejor vendedor?

📚 Documentos recuperados (top 3):
  [1] (dist=18.71) Venta del 18 de January del 2024: Se vendieron 10 unidades de 'Teclado Mecánico'...
  [2] (dist=18.73) Venta del 15 de February del 2024: Se vendieron 30 unidades de 'Mouse Inalámbric...
  [3] (dist=19.06) Venta del 16 de January del 2024: Se vendieron 25 unidades de 'Mouse Inalámbrico...

🤖 Respuesta del Bot:
🏆 El mejor vendedor es **Ana López** con ingresos de **$14,450.00**.

❓ Pregunta: ¿Cuá

## 🎯 Paso 8: Modo interactivo — escribe tu propia pregunta

In [ ]:
# Modo interactivo
print("🤖 Bot RAG de Ventas - Modo Interactivo")
print("Escribe 'salir' para terminar\n")

while True:
    pregunta = input("\n❓ Tu pregunta: ").strip()
    if not pregunta:
        continue
    if pregunta.lower() in ['salir', 'exit', 'quit']:
        print("👋 ¡Hasta luego!")
        break
    bot.preguntar(pregunta)

🤖 Bot RAG de Ventas - Modo Interactivo
Escribe 'salir' para terminar


❓ Tu pregunta: que se vendio mas 

❓ Pregunta: que se vendio mas

📚 Documentos recuperados (top 3):
  [1] (dist=8.81) Venta del 15 de February del 2024: Se vendieron 30 unidades de 'Mouse Inalámbric...
  [2] (dist=8.99) Venta del 16 de January del 2024: Se vendieron 25 unidades de 'Mouse Inalámbrico...
  [3] (dist=9.10) Venta del 18 de January del 2024: Se vendieron 10 unidades de 'Teclado Mecánico'...

🤖 Respuesta del Bot:
🔍 **Basado en los datos más relevantes encontrados:**

- Venta del 15 de February del 2024: Se vendieron 30 unidades de 'Mouse Inalámbrico' (categoría Accesorios) a $29.99 cada una. Total de la venta: $899.70. Vendedor: Carlos Ruiz, región: Sur. Trimestre: Q1.
- Venta del 16 de January del 2024: Se vendieron 25 unidades de 'Mouse Inalámbrico' (categoría Accesorios) a $29.99 cada una. Total de la venta: $749.75. Vendedor: Carlos Ruiz, región: Sur. Trimestre: Q1.
- Venta del 18 de January del 2024:

## 📈 Paso 9 (Opcional): Visualizaciones con matplotlib

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Dashboard de Ventas', fontsize=16, fontweight='bold')

# Ingresos por producto
prod = df_norm.groupby('producto')['total_venta'].sum().sort_values(ascending=True)
prod.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Ingresos por Producto')
axes[0].set_xlabel('Ingresos ($)')

# Ingresos por vendedor
vend = df_norm.groupby('vendedor')['total_venta'].sum().sort_values(ascending=True)
vend.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Ingresos por Vendedor')
axes[1].set_xlabel('Ingresos ($)')

# Ingresos por región
reg = df_norm.groupby('region')['total_venta'].sum()
reg.plot(kind='pie', ax=axes[2], autopct='%1.1f%%', startangle=90)
axes[2].set_title('Distribución por Región')
axes[2].set_ylabel('')

plt.tight_layout()
plt.savefig('dashboard_ventas.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard guardado como 'dashboard_ventas.png'")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Agrupar ventas totales por trimestre
ventas_por_trimestre = df_norm.groupby('trimestre')['total_venta'].sum().reset_index()

# Asegurarse de que los trimestres estén en el orden correcto para la visualización
orden_trimestres = ['Q1', 'Q2', 'Q3', 'Q4']
ventas_por_trimestre['trimestre'] = pd.Categorical(ventas_por_trimestre['trimestre'], categories=orden_trimestres, ordered=True)
ventas_por_trimestre = ventas_por_trimestre.sort_values('trimestre')

# Crear el gráfico de barras
plt.figure(figsize=(8, 6))
sns.barplot(x='trimestre', y='total_venta', data=ventas_por_trimestre, palette='viridis', hue='trimestre', legend=False)
plt.title('Ventas Totales por Trimestre', fontsize=16)
plt.xlabel('Trimestre', fontsize=12)
plt.ylabel('Ventas Totales ($)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Crear la columna 'semestre'
df_norm['semestre'] = df_norm['fecha'].dt.month.apply(lambda x: 'Primer Semestre' if x <= 6 else 'Segundo Semestre')

# Agrupar ventas totales por semestre
ventas_por_semestre = df_norm.groupby('semestre')['total_venta'].sum().reset_index()

# Definir el orden de los semestres para la visualización
orden_semestres = ['Primer Semestre', 'Segundo Semestre']
ventas_por_semestre['semestre'] = pd.Categorical(ventas_por_semestre['semestre'], categories=orden_semestres, ordered=True)
ventas_por_semestre = ventas_por_semestre.sort_values('semestre')

# Crear el gráfico de barras para ventas por semestre
plt.figure(figsize=(8, 6))
sns.barplot(x='semestre', y='total_venta', data=ventas_por_semestre, palette='coolwarm', hue='semestre', legend=False)
plt.title('Ventas Totales por Semestre', fontsize=16)
plt.xlabel('Semestre', fontsize=12)
plt.ylabel('Ventas Totales ($)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

---
## 🗺️ Resumen de la arquitectura

```
ventas.csv
    │
    ▼
NormalizadorVentas (Pandas)
  ├── Limpieza (nulos, duplicados)
  ├── Enriquecimiento (total_venta, mes, trimestre)
  ├── MinMaxScaler (numéricas)
  └── LabelEncoder (categóricas)
    │
    ▼
GeneradorCorpus
  ├── Documentos por fila
  ├── Resumen por producto
  └── Resumen por vendedor
    │
    ▼
SentenceTransformer → Embeddings
    │
    ▼
FAISS Index (búsqueda vectorial)
    │
    ▼
BotRAG.preguntar(query)
  ├── Recuperar top-k docs
  └── Generar respuesta
```

### 🔧 Para usar con tu propio archivo ventas.csv:
```python
# Reemplaza la celda de datos por:
df = pd.read_csv('/content/ventas.csv')  # sube tu archivo a Colab
```
